# 🛠️ Chapter 05-03b: PROJECT — Enterprise RAG Knowledge Bot

---

## 🎯 Build a Production-Ready RAG System!

This is your first mini-project. You'll build a complete RAG system with evaluation.

**Requirements:**
- ✅ Ingest multiple document types
- ✅ Smart chunking with metadata
- ✅ Advanced retrieval (hybrid + re-rank)
- ✅ Quality evaluation
- ✅ Conversation memory

Let's build! 🚀

---

## 📦 Setup

In [ ]:
# !pip install langchain-openai langchain-community langchain-core chromadb python-dotenv rank_bm25

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, AIMessage

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print("✅ Ready!")

---

## 📄 Step 1: Build the Knowledge Base

In [ ]:
# Simulated enterprise documents from different departments

hr_docs = [
    "Vacation Policy: All full-time employees receive 20 days of PTO per year. Unused days carry over up to 5 days. Part-time employees receive pro-rated PTO.",
    "Remote Work: Employees may work remotely up to 3 days per week with manager approval. Full remote requires VP approval. All remote workers must be available during core hours 10 AM - 3 PM EST.",
    "Benefits: Health insurance (medical, dental, vision) is provided for all employees starting day 1. 401(k) matching up to 4%. Annual learning budget of $2,000 per employee.",
    "Performance Reviews: Conducted quarterly. Annual promotions considered in Q4 review cycle. Self-assessment due 2 weeks before review date.",
]

engineering_docs = [
    "Tech Stack: Backend: Python (FastAPI), Database: PostgreSQL + Redis, Frontend: React + TypeScript, Infrastructure: AWS (ECS, RDS, S3), CI/CD: GitHub Actions.",
    "Code Review Policy: All PRs require 2 approvals. PRs must include tests. No direct pushes to main branch. Hotfixes require tech lead approval.",
    "On-Call Rotation: Engineers rotate weekly. Primary on-call responds within 15 minutes. Escalation to secondary after 30 minutes. Post-incident review within 48 hours.",
    "Architecture: Microservices with event-driven communication via AWS SNS/SQS. API Gateway handles authentication. Each service owns its database (database-per-service pattern).",
]

product_docs = [
    "DataPro Basic ($99/mo): Up to 10 users, 100GB storage, email support, standard analytics dashboard.",
    "DataPro Pro ($199/mo): Up to 50 users, 1TB storage, priority support, advanced analytics, API access (1000 req/min).",
    "DataPro Enterprise (custom): Unlimited users, unlimited storage, dedicated support, SSO/SAML, custom integrations, 99.99% SLA.",
    "Roadmap Q1 2025: AI-powered anomaly detection, real-time collaboration features, mobile app v2, Snowflake integration.",
]

# Create documents with metadata
all_docs = []
for doc_text in hr_docs:
    all_docs.append(Document(page_content=doc_text, metadata={"department": "HR", "type": "policy"}))
for doc_text in engineering_docs:
    all_docs.append(Document(page_content=doc_text, metadata={"department": "Engineering", "type": "technical"}))
for doc_text in product_docs:
    all_docs.append(Document(page_content=doc_text, metadata={"department": "Product", "type": "product"}))

print(f"📄 Total documents: {len(all_docs)}")
for dept in ["HR", "Engineering", "Product"]:
    count = sum(1 for d in all_docs if d.metadata["department"] == dept)
    print(f"   {dept}: {count} docs")

In [ ]:
# Split and store
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(all_docs)

vectorstore = Chroma.from_documents(chunks, embeddings, collection_name="enterprise_kb")
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print(f"✅ {len(chunks)} chunks stored in vector DB")

---

## 🤖 Step 2: Build the RAG Bot with Memory

In [ ]:
# RAG bot with conversation history

system_prompt = """You are an enterprise knowledge assistant for DataPro Inc.
Answer questions based ONLY on the provided context.
If the context doesn't contain the answer, say "I don't have that information in my knowledge base."
Be specific, cite department/source when possible.
Keep answers concise but complete."""

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", """Context from knowledge base:
{context}

Question: {question}""")
])

class EnterpriseRAGBot:
    def __init__(self):
        self.chat_history = []
        self.format_docs = lambda docs: "\n\n---\n\n".join(
            f"[{d.metadata.get('department', 'Unknown')}] {d.page_content}" for d in docs
        )
    
    def ask(self, question: str, department_filter: str = None) -> str:
        # Retrieve with optional department filter
        if department_filter:
            results = vectorstore.similarity_search(
                question, k=4, filter={"department": department_filter}
            )
        else:
            results = retriever.invoke(question)
        
        context = self.format_docs(results)
        
        # Generate answer with conversation history
        chain = rag_prompt | llm | StrOutputParser()
        answer = chain.invoke({
            "context": context,
            "question": question,
            "chat_history": self.chat_history
        })
        
        # Update history
        self.chat_history.append(HumanMessage(content=question))
        self.chat_history.append(AIMessage(content=answer))
        
        return answer, results

bot = EnterpriseRAGBot()
print("✅ Enterprise RAG Bot ready!")

In [ ]:
# Test the bot
test_questions = [
    "How many vacation days do employees get?",
    "What about part-time employees?",  # Follow-up using context
    "What's the tech stack?",
    "How much does the Pro plan cost?",
]

for q in test_questions:
    print(f"❓ {q}")
    answer, sources = bot.ask(q)
    print(f"💬 {answer}")
    print(f"📎 Sources: {[s.metadata['department'] for s in sources]}")
    print("─" * 60)

---

## 📊 Step 3: Evaluate Quality

In [ ]:
# Simple evaluation framework (without RAGAS dependency)

eval_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an evaluation judge. Score the answer on three criteria:
1. Faithfulness (0-10): Is the answer supported by the context? No made-up info?
2. Relevance (0-10): Does the answer address the question?
3. Completeness (0-10): Is the answer thorough?

Return ONLY a JSON object: {{"faithfulness": X, "relevance": X, "completeness": X, "explanation": "brief note"}}"""),
    ("human", "Question: {question}\nContext: {context}\nAnswer: {answer}")
])

from langchain_core.output_parsers import JsonOutputParser

eval_chain = eval_prompt | llm | JsonOutputParser()

# Evaluation dataset
eval_set = [
    {"question": "How many PTO days do full-time employees get?", "expected": "20 days"},
    {"question": "What database does DataPro use?", "expected": "PostgreSQL + Redis"},
    {"question": "What's included in the Enterprise plan?", "expected": "Unlimited users, unlimited storage, dedicated support, SSO"},
]

# Reset bot for clean evaluation
eval_bot = EnterpriseRAGBot()

print("📊 RAG Quality Evaluation")
print("=" * 60)

total_scores = {"faithfulness": 0, "relevance": 0, "completeness": 0}

for item in eval_set:
    answer, sources = eval_bot.ask(item["question"])
    context = "\n".join(s.page_content for s in sources)
    
    scores = eval_chain.invoke({
        "question": item["question"],
        "context": context,
        "answer": answer
    })
    
    print(f"\n❓ {item['question']}")
    print(f"💬 {answer[:100]}...")
    print(f"📏 Faithfulness: {scores['faithfulness']}/10 | Relevance: {scores['relevance']}/10 | Completeness: {scores['completeness']}/10")
    
    for key in total_scores:
        total_scores[key] += scores[key]

print(f"\n{'='*60}")
n = len(eval_set)
print(f"📊 Average Scores:")
for key, total in total_scores.items():
    avg = total / n
    bar = "█" * int(avg)
    print(f"   {key:15s}: {avg:.1f}/10 {bar}")

---

## ✍️ Practice: Add Department-Scoped Search

**Challenge:** Add the ability to restrict searches to a specific department.

In [ ]:
# YOUR CODE HERE

# Test department-scoped queries:
# bot.ask("What policies apply to me?", department_filter="HR")
# bot.ask("What's our deployment process?", department_filter="Engineering")

<details>
<summary>💡 Click to see solution</summary>

```python
# The bot already supports this!
scoped_bot = EnterpriseRAGBot()

answer, sources = scoped_bot.ask("What policies apply to me?", department_filter="HR")
print(f"HR only: {answer}")
print(f"Sources: {[s.metadata['department'] for s in sources]}")

answer, sources = scoped_bot.ask("How do deployments work?", department_filter="Engineering")
print(f"\nEng only: {answer}")
print(f"Sources: {[s.metadata['department'] for s in sources]}")
```

</details>

---

## ✅ Key Takeaways

```
┌─────────────────────────────────────────────────────────────────────┐
│  📌 Enterprise RAG = metadata filtering + conversation memory.     │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Evaluate with Faithfulness, Relevance, Completeness.           │
├─────────────────────────────────────────────────────────────────────┤
│  📌 LLM-as-judge works for rapid evaluation during development.    │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Department-scoped search = multi-tenant access control.        │
├─────────────────────────────────────────────────────────────────────┤
│  📌 You've completed the full RAG journey: basic → advanced → prod!│
└─────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: AI Agents Fundamentals!

Chapter 6: From chains to agents — LLMs that can reason, plan, and act.

---

### 🎉 You Built Enterprise RAG!
Multi-department, conversation memory, quality evaluation — portfolio-worthy project. 🚀